# Tests: `fastermodels.card` (source `nbs/02_card.ipynb`)

In [ ]:
from fastcore.test import *
from fastermodels.card import FORBIDDEN, check_card, render_card

In [ ]:
_meta = dict(
    name='test-resnet18-imagenette', base_model='torchvision/resnet18', license='bsd-3-clause',
    datasets=['frgfm/imagenette'], tags=['fasterai', 'pruning'],
    scope_line='Imagenette, n=3925, Wilson half-width about 1 pt; pipeline evidence, not a published claim.',
    input_shape='3x160x160',
    recipe={'prune': 'ratio 0.3, local, round_to 8', 'recovery': '3 epochs'},
    reference={'name': 'resnet18 fine-tuned on Imagenette', 'k': 3700, 'n': 3925, 'bytes': 44_726_568,
               'params': 11_181_642, 'macs': 1_824_000_000, 'peak_activation_bytes': 3_211_264},
    rows=[dict(artifact='pruned FP32', file='model.safetensors', params=8_900_000, bytes=35_600_000,
               macs=912_000_000, peak_activation_bytes=2_408_448,
               k=3680, n=3925, delta=-0.51, lo=-1.2, hi=0.2, p_mcnemar=0.12)],
    latency=None,
    provenance={'fasterai': '0.4.0', 'fastermodels': '0.1.0', 'torch': '2.9.1', 'measured_on': '2026-09-11'})

_card = render_card(_meta)

# a complete card says nothing a reader has to take on trust
test_eq(check_card(_card), [])

# the front matter the Hub reads
assert _card.startswith('---\n')
assert 'library_name: fastermodels' in _card
assert 'license: bsd-3-clause' in _card
assert 'base_model: torchvision/resnet18' in _card
assert '  - frgfm/imagenette' in _card

# the four criteria, each with the reference, the artifact and the gap between them
assert '| criterion | reference | this artifact | gap |' in _card
assert '| top-1 | 3700/3925 = 94.27 % (Wilson 95 % [93.50, 94.95]) | 3680/3925 = 93.76 %' in _card
assert '-0.51 pt, 95 % CI [-1.20, +0.20], McNemar p 0.1200' in _card
assert '| size | 44,726,568 B, 11,181,642 params | 35,600,000 B, 8,900,000 params | -20.4 % bytes, -20.4 % params |' in _card
assert '| memory | 3,211,264 B | 2,408,448 B | -25.0 % |' in _card

# half the reference's MACs reads -50.0 %, and never as a ratio the speedup rule would flag
assert '| MACs | 1,824,000,000 | 912,000,000 | -50.0 % |' in _card
assert '0.5x' not in _card and '2x' not in _card

# the resolution the memory was measured at is named
assert '3x160x160' in _card

# a latency that was not measured says so, it never reads as a zero
assert 'non mesurée' in _card
assert '0.00 ms' not in _card

# a reference value the producer did not measure reads n/a, and the card stays clean
_partial = render_card({**_meta, 'reference': {'name': 'source', 'k': 3700, 'n': 3925}})
assert '| MACs | n/a | 912,000,000 | n/a |' in _partial
test_eq(check_card(_partial), [])

# the accuracy target is reported under the table, met or not, and only when the row carries one
assert 'Accuracy target' not in _card
_met = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'target': -2.0, 'lo': -1.71}]})
assert 'Accuracy target: -2.0 pt — met (lower bound -1.71)' in _met
_missed = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'target': -2.0, 'lo': -2.14}]})
assert 'Accuracy target: -2.0 pt — not met (lower bound -2.14)' in _missed
test_eq(check_card(_met), [])
test_eq(check_card(_missed), [])

# the ladder lets a reader pick another point on it; without one the card is unchanged
_ladder = render_card({**_meta, 'ladder': [
    {'name': 'pruned INT8', 'repo': 'FasterAI-Labs/resnet18-int8', 'delta': -1.4,
     'bytes': 11_200_000, 'peak_activation_bytes': 602_112, 'macs': 1_368_000_000},
    {'name': 'pruned 50 %', 'repo': 'FasterAI-Labs/resnet18-p50', 'delta': -3.1,
     'bytes': 22_300_000, 'peak_activation_bytes': 1_605_632, 'macs': 912_000_000}]})
assert '## Variants' in _ladder
assert '| variant | repo | top-1 gap (pt) | bytes | memory (B) | MACs |' in _ladder
assert '| pruned INT8 | `FasterAI-Labs/resnet18-int8` | -1.40 | 11,200,000 | 602,112 | 1,368,000,000 |' in _ladder
test_eq(check_card(_ladder), [])
assert '## Variants' not in _card
test_eq(render_card({**_meta, 'ladder': []}), _card)

# the card compares to something named, on a stated number of images, or it refuses to render
for _f in ('name', 'k', 'n'):
    with ExceptionExpected(KeyError, regex=_f):
        render_card({**_meta, 'reference': {k: v for k, v in _meta['reference'].items() if k != _f}})

# parity is a gate matter: agreement is not on the card, and an old meta carrying it still renders
_old = render_card({**_meta, 'rows': [{**_meta['rows'][0], 'agreement': 1.0, 'agreement_kind': 'same-precision'}]})
test_eq(_old, _card)
assert 'agreement' not in _card

In [ ]:
# a measured latency is written with the device, the runtime, the precision and the batch
_measured = render_card({**_meta, 'latency': [dict(device='Jetson Orin NX', runtime='tensorrt', precision='fp16',
                                                   batch=1, median_ms=0.507, n_runs=100)]})
assert 'non mesurée' not in _measured
assert 'Jetson Orin NX' in _measured and 'tensorrt' in _measured and '0.507' in _measured
test_eq(check_card(_measured), [])

In [ ]:
# every forbidden phrase is reported, once, wherever it is injected
for _p in FORBIDDEN:
    test_eq(check_card(_card + f'\nThe artifact is {_p} on this line.\n'), [_p])
    test_eq(check_card(_card + f'\nThe artifact is {_p.upper()} on this line.\n'), [_p])

# whole words only: a phrase inside a longer word is not a claim
test_eq(check_card('the nanometre scale'), [])
test_eq(check_card('todolist'), [])

In [ ]:
# a speedup with neither device nor runtime on its line is reported
_flagged = check_card('the model is 2.3x faster')
test_eq(len(_flagged), 1)
assert '2.3x' in _flagged[0]
test_eq(len(check_card('the model is 2.3× faster')), 1)
test_eq(len(check_card('3x faster')), 1)

# a device alone, or a runtime alone, still leaves the claim unreadable
test_eq(len(check_card('the model is 2.3x faster on CPU')), 1)
test_eq(len(check_card('the model is 2.3x faster with onnxruntime')), 1)

# the same claim with both the device and the runtime it was measured on is not reported
test_eq(check_card('the model is 2.3x on CPU with onnxruntime'), [])
test_eq(check_card('2.3x vs the FP32 engine on a Jetson Orin NX with tensorrt, batch 1'), [])

# a number that is not a speedup is not a claim
test_eq(check_card('a 1x1 convolution'), [])
test_eq(check_card('the matrix is 3x4'), [])